# Engineered Features Comparison: Train.csv vs Test.csv

This notebook compares the training and test datasets using engineered features (431 features) instead of raw features.
This analysis helps us understand what possible shifts the model is dealing with after feature engineering.

We will:
1. Load both datasets
2. Apply feature engineering to both datasets
3. Handle missing values (-9999 in test data)
4. Compare statistical distributions of engineered features
5. Perform statistical tests to assess similarity
6. Visualize differences

**Note**: The test dataset contains -9999 values which represent missing data and should be treated as NaN.


In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.backends.backend_pdf import PdfPages
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Add project root to path to import feature engineering module
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))
from aquaculture.feature_engineering import AquacultureFeatureEngineer
from aquaculture.feature_selection import FeatureSelector

In [2]:
# Specify experiment directory to load SHAP feature importance from
# User should modify this path to point to their desired experiment
experiment_dir = '../experiments/20260810_113750'  # Example - modify as needed


In [3]:
# Load the datasets
print("Loading datasets...")
train_df = pd.read_csv('../data/Train.csv')
test_df = pd.read_csv('../data/Test.csv')

print(f"Training dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")

Loading datasets...
Training dataset shape: (1821, 146)
Test dataset shape: (1030, 145)


In [4]:
# Prepare raw data for feature engineering
# For training data: exclude ID and label columns
# For test data: exclude ID column

train_feature_cols = [col for col in train_df.columns if col not in ['ID', 'label']]
test_feature_cols = [col for col in test_df.columns if col != 'ID']

X_train_raw = train_df[train_feature_cols].values
X_test_raw = test_df[test_feature_cols].values

# Reshape to 3D as expected by the feature engineer: (n_samples, 12, 12)
X_train = X_train_raw.reshape(X_train_raw.shape[0], 12, 12)
X_test = X_test_raw.reshape(X_test_raw.shape[0], 12, 12)

print(f"Reshaped training data shape: {X_train.shape}")
print(f"Reshaped test data shape: {X_test.shape}")

Reshaped training data shape: (1821, 12, 12)
Reshaped test data shape: (1030, 12, 12)


In [5]:
# Create and fit feature engineer
print("Creating and fitting feature engineer...")
feature_engineer = AquacultureFeatureEngineer(
    simulate_mask=False,  # No stochastic masking for fair comparison
    random_state=42,
    include_optical=True,      # Set to False to exclude optical features
    include_sar=True,          # Set to False to exclude SAR features
    include_cross_sensor_features=True,  # Set to False to exclude cross-sensor features
    include_temporal_statistics=True,    # Set to False to exclude temporal statistics
    include_normalized_optical=False,     # Set to False to exclude normalized optical features
    include_directional_vote=True,     # Set to False to exclude directional vote features
    include_metadata=False      # Set to False to exclude metadata features
)

# Fit on training data to establish feature names FIRST (required for FeatureSelector)
print("Fitting feature engineer on training data...")
try:
    feature_engineer.fit(X_train)
    print("✓ Fitted feature engineer on training data")
except Exception as e:
    print(f"⚠ Error fitting feature engineer: {e}")

Creating and fitting feature engineer...
Fitting feature engineer on training data...
✓ Fitted feature engineer on training data


In [ ]:
# Configure feature selection
feature_selector = None  # Initialize to None
#feature_selector = FeatureSelector(
#    feature_engineer,
#    selection_method='groups',
#    groups=['temporal', 'metadata']  # Only temporal and metadata features
#)
    #'include': {                      # Start with all features
    #    'method': 'patterns',
    #    'patterns': ['.*']            # Match everything
    #},

feature_selector = FeatureSelector(
    base_engineer=feature_engineer,
    selection_method='combine',
    include={
        'method': 'groups',
        'groups': ['temporal', 'directional_vote']
    },
    exclude={                      # Then exclude the unwanted prefixes
        'method': 'patterns',
        'patterns': [
            'green_.*',   # Features starting with green_
            'nir_.*',     # Features starting with nir_
            'nira_.*',    # Features starting with nira_
            'swir1_.*',   # Features starting with swir1_
            'swir2_.*'    # Features starting with swir2_
        ]
    }
)
print("✓ Configured feature selector.")



# Transform both datasets
print("Transforming training data...")
try:
    # Use the feature selector on RAW data - it will internally use the feature engineer
    X_train_features = feature_selector.transform(X_train, training=False)
    print(f"✓ Transformed and selected training data: {X_train_features.shape[1]} features")
except Exception as e:
    print(f"⚠ Error transforming training data: {e}")
    # Fallback to just feature engineering
    X_train_features = feature_engineer.transform(X_train, training=False)
    print(f"✓ Fallback: Transformed training data: {X_train_features.shape[1]} features (no selection)")

print("Transforming test data...")
try:
    # Use the same fitted transformer for test data
    X_test_features = feature_selector.transform(X_test, training=False)
    print(f"✓ Transformed and selected test data: {X_test_features.shape[1]} features")
except Exception as e:
    print(f"⚠ Error transforming test data: {e}")
    # Fallback to just feature engineering
    X_test_features = feature_engineer.transform(X_test, training=False)
    print(f"✓ Fallback: Transformed test data: {X_test_features.shape[1]} features (no selection)")

# Get feature names
try:
    if hasattr(X_train_features, 'columns'):
        feature_names = list(X_train_features.columns)
    elif hasattr(feature_selector, 'get_feature_names_out'):
        # Try to get feature names from the selector
        try:
            feature_names = list(feature_selector.get_feature_names_out())
        except:
            feature_names = [f"feature_{i}" for i in range(X_train_features.shape[1])]
    elif hasattr(feature_engineer, 'get_feature_names_out'):
        # Try to get feature names from the engineer
        feature_names = list(feature_engineer.get_feature_names_out())
    else:
        feature_names = [f"feature_{i}" for i in range(X_train_features.shape[1])]
except:
    feature_names = [f"feature_{i}" for i in range(X_train_features.shape[1])]

# Convert to DataFrames for easier analysis
try:
    train_features_df = pd.DataFrame(X_train_features.values, columns=feature_names)
    test_features_df = pd.DataFrame(X_test_features.values, columns=feature_names)
    print(f"Engineered features shape - Train: {train_features_df.shape}")
    print(f"Engineered features shape - Test: {test_features_df.shape}")
    print(f"Number of features: {len(feature_names)}")
    
    # Save the dataframes to CSV for further analysis
    train_features_df.to_csv('../data/engineered_train_features.csv', index=False)
    test_features_df.to_csv('../data/engineered_test_features.csv', index=False)
except Exception as e:
    print(f"⚠ Error creating DataFrames or saving CSV: {e}")


SyntaxError: positional argument follows keyword argument (2665728901.py, line 16)

In [ ]:
# Check for missing values (-9999 in test data that became NaN after feature engineering)
print("Checking for missing values after feature engineering...")

train_missing = train_features_df.isnull().sum().sum()
test_missing = test_features_df.isnull().sum().sum()

print(f"Missing values in training features: {train_missing}")
print(f"Missing values in test features: {test_missing}")

# Show columns with missing values in test data (if any)
if test_missing > 0:
    missing_cols = test_features_df.columns[test_features_df.isnull().any()].tolist()
    print(f"\nColumns with missing values in test data ({len(missing_cols)} columns):")
    for i, col in enumerate(missing_cols[:10]):  # Show first 10
        missing_count = test_features_df[col].isnull().sum()
        print(f"  {col}: {missing_count} missing values")
    if len(missing_cols) > 10:
        print(f"  ... and {len(missing_cols) - 10} more")
else:
    print("\n✓ No missing values found in test features")

## Basic Statistics Comparison

Let's compare basic statistics (mean, median, std) for each engineered feature between train and test datasets.

In [ ]:
# Calculate basic statistics
print("Calculating basic statistics...")

# Compute statistics for training data
train_stats = train_features_df.describe()

# Compute statistics for test data (ignoring NaN values)
test_stats = test_features_df.describe()

# Create a comparison dataframe
comparison_stats = pd.DataFrame()
comparison_stats['train_mean'] = train_stats.loc['mean']
comparison_stats['test_mean'] = test_stats.loc['mean']
comparison_stats['train_std'] = train_stats.loc['std']
comparison_stats['test_std'] = test_stats.loc['std']
comparison_stats['train_median'] = train_features_df.median()
comparison_stats['test_median'] = test_features_df.median()

# Calculate differences
comparison_stats['mean_diff'] = comparison_stats['test_mean'] - comparison_stats['train_mean']
comparison_stats['mean_diff_pct'] = (comparison_stats['mean_diff'] / comparison_stats['train_mean'].abs()) * 100
comparison_stats['std_ratio'] = comparison_stats['test_std'] / comparison_stats['train_std']

# Replace infinite values from division by zero
comparison_stats['mean_diff_pct'] = comparison_stats['mean_diff_pct'].replace([np.inf, -np.inf], np.nan)
comparison_stats['std_ratio'] = comparison_stats['std_ratio'].replace([np.inf, -np.inf], np.nan)

print("\nComparison statistics (first 10 features):")
display(comparison_stats.head(10))

# Summary of differences
print(f"\nMean difference statistics:")
print(f"  Mean absolute difference: {comparison_stats['mean_diff'].abs().mean():.4f}")
print(f"  Median absolute difference: {comparison_stats['mean_diff'].abs().median():.4f}")
print(f"  Max absolute difference: {comparison_stats['mean_diff'].abs().max():.4f}")

print(f"\nMean difference percentage statistics:")
print(f"  Mean absolute % difference: {np.nanmean(np.abs(comparison_stats['mean_diff_pct'])):.2f}%")
print(f"  Median absolute % difference: {np.nanmedian(np.abs(comparison_stats['mean_diff_pct'])):.2f}%")

print(f"\nStd ratio statistics:")
print(f"  Mean std ratio: {np.nanmean(comparison_stats['std_ratio']):.4f}")
print(f"  Median std ratio: {np.nanmedian(comparison_stats['std_ratio']):.4f}")

# Save comparison statistics to CSV for further analysis
comparison_stats.to_csv('../data/engineered_features_comparison_stats.csv', index=True)

## Statistical Tests for Distribution Similarity

Beyond basic statistics, we can use statistical tests to evaluate whether the distributions are significantly different.

In [ ]:
# Statistical tests for distribution comparison
print("Running statistical tests for distribution similarity...")

# Initialize results list
test_results = []

# For each feature, perform KS test and t-test
for col in train_features_df.columns:
    # Get non-NaN values for both datasets
    train_vals = train_features_df[col].dropna().values
    test_vals = test_features_df[col].dropna().values
    
    # Skip if insufficient data
    if len(train_vals) < 2 or len(test_vals) < 2:
        continue
    
    # Kolmogorov-Smirnov test (compares distributions)
    ks_stat, ks_pvalue = stats.ks_2samp(train_vals, test_vals)
    
    # T-test for difference in means (Welch's t-test for unequal variances)
    t_stat, t_pvalue = stats.ttest_ind(train_vals, test_vals, equal_var=False)
    
    # Calculate Cohen's d (effect size)
    pooled_std = np.sqrt(((len(train_vals)-1)*np.var(train_vals, ddof=1) + (len(test_vals)-1)*np.var(test_vals, ddof=1)) / (len(train_vals) + len(test_vals) - 2))
    if pooled_std > 0:
        cohens_d = (np.mean(test_vals) - np.mean(train_vals)) / pooled_std
    else:
        cohens_d = 0
    
    # Store results
    test_results.append({
        'feature': col,
        'ks_statistic': ks_stat,
        'ks_pvalue': ks_pvalue,
        't_statistic': t_stat,
        't_pvalue': t_pvalue,
        'cohens_d': cohens_d,
        'train_mean': np.mean(train_vals),
        'test_mean': np.mean(test_vals),
        'train_std': np.std(train_vals, ddof=1),
        'test_std': np.std(test_vals, ddof=1)
    })

# Convert to DataFrame
results_df = pd.DataFrame(test_results)

# Summary of test results
print(f"\nKolmogorov-Smirnov Test Results:")
print(f"  Features with p < 0.05 (significantly different distributions): {(results_df['ks_pvalue'] < 0.05).sum()} out of {len(results_df)}")
print(f"  Median KS statistic: {results_df['ks_statistic'].median():.4f}")

print(f"\nT-test Results (Mean Differences):")
print(f"  Features with p < 0.05 (significantly different means): {(results_df['t_pvalue'] < 0.05).sum()} out of {len(results_df)}")
print(f"  Median |Cohen's d|: {np.abs(results_df['cohens_d']).median():.4f}")

print(f"\nEffect Size Interpretation (Cohen's d):")
small_effect = (np.abs(results_df['cohens_d']) < 0.2).sum()
medium_effect = ((np.abs(results_df['cohens_d']) >= 0.2) & (np.abs(results_df['cohens_d']) < 0.5)).sum()
large_effect = (np.abs(results_df['cohens_d']) >= 0.5).sum()
print(f"  Small effect (<0.2): {small_effect} features")
print(f"  Medium effect (0.2-0.5): {medium_effect} features")
print(f"  Large effect (>0.5): {large_effect} features")

print(f"\nDetailed results (first 10 features):")
display(results_df[['feature', 'ks_pvalue', 't_pvalue', 'cohens_d']].head(10))


## Population Stability Index (PSI) Analysis

Population Stability Index (PSI) is a metric commonly used in machine learning to detect shifts in population distributions. It's particularly useful for monitoring data drift.

PSI Formula: PSI = Σ((% Actual - % Expected) * ln(% Actual / % Expected))

General guidelines:
- PSI < 0.1: No significant change
- 0.1 ≤ PSI < 0.2: Moderate change
- PSI ≥ 0.2: Significant change (potential data drift)


In [ ]:
# Calculate Population Stability Index (PSI) for each feature
print("Calculating Population Stability Index (PSI)...")

def calculate_psi(expected, actual, buckets=10):
    """Calculate Population Stability Index (PSI)."""
    # Remove any infinite or NaN values
    expected = expected[np.isfinite(expected)]
    actual = actual[np.isfinite(actual)]

    if len(expected) == 0 or len(actual) == 0:
        return np.nan

    # Check unique values
    expected_unique = np.unique(expected)
    actual_unique = np.unique(actual)
    n_exp_unique = len(expected_unique)
    n_act_unique = len(actual_unique)

    # Special case: Detect z-score derived temporal statistics
    # These theoretically should be constant or near-constant
    is_z_mean_feature = np.allclose(expected, 0.0, atol=1e-10) and np.allclose(actual, 0.0, atol=1e-10)
    is_z_std_feature = (np.allclose(expected, 1.0, atol=1e-10) or np.allclose(expected, 0.0, atol=1e-10)) and \
                       (np.allclose(actual, 1.0, atol=1e-10) or np.allclose(actual, 0.0, atol=1e-10))

    # Start with the requested number of buckets and reduce if needed
    for trial_buckets in range(min(buckets, max(n_exp_unique, n_act_unique)), 1, -1):
        # We need at least 2 bins to make a histogram meaningful
        if trial_buckets < 10:
            # Not enough variation for any meaningful comparison
            if is_z_mean_feature or is_z_std_feature:
                return 0.0  # Essentially identical (no variation to compare)
            else:
                return np.nan  # Not enough variation for meaningful comparison
        
        try:
            # Try to create bins
            _, bin_edges = np.histogram(expected, bins=trial_buckets)
            # If we get here, binning succeeded
            effective_buckets = trial_buckets
            break
        except ValueError as e:
            if "Too many bins for data range" in str(e):
                # Try with fewer bins
                continue
            else:
                # Re-raise unexpected errors
                raise
    else:
        # Loop completed without breaking - no valid bucket count found
        if is_z_mean_feature or is_z_std_feature:
            return 0.0  # Essentially identical
        else:
            return np.nan  # Not enough variation

    # Create bins and calculate PSI as before
    _, bin_edges = np.histogram(expected, bins=effective_buckets)
    expected_counts, _ = np.histogram(expected, bins=bin_edges)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)

    expected_perc = expected_counts / len(expected)
    actual_perc = actual_counts / len(actual)

    # Avoid division by zero by adding a small epsilon
    epsilon = 1e-10
    expected_perc = np.maximum(expected_perc, epsilon)
    actual_perc = np.maximum(actual_perc, epsilon)

    # Calculate PSI
    psi = np.sum((actual_perc - expected_perc) * np.log(actual_perc / expected_perc))
    return psi

# Calculate PSI for each feature
psi_results = []

for col in train_features_df.columns:
    # Get non-NaN values
    train_vals = train_features_df[col].dropna().values
    test_vals = test_features_df[col].dropna().values
    
    if len(train_vals) > 0 and len(test_vals) > 0:
        psi = calculate_psi(train_vals, test_vals, buckets=10)
        psi_results.append({
            "feature": col,
            "psi": psi
        })
    else:
        psi_results.append({
            "feature": col,
            "psi": np.nan
        })

# Convert to DataFrame and sort by PSI (descending)
psi_df = pd.DataFrame(psi_results)
psi_df = psi_df.sort_values("psi", ascending=False)


## Visualization of Distributions - All features

Let's visualize the distributions of ALL engineered features to better understand the differences between train and test datasets, and save the plots to a PDF file.


In [ ]:
# Create a PDF with distribution plots for all features
print("Creating PDF with distribution plots for all features...")

def safe_histogram(ax, data, label, color, alpha=0.7):
    """Safely plot histogram handling edge cases like constant values or too many bins."""
    # Remove NaN/inf values
    clean_data = data[np.isfinite(data)]

    if len(clean_data) == 0:
        # No valid data - show empty plot with message
        ax.text(0.5, 0.5, 'No Data', transform=ax.transAxes,
                ha='center', va='center', fontsize=10)
        return

    # Check if all values are identical (or nearly identical)
    data_range = np.max(clean_data) - np.min(clean_data)
    if data_range < 1e-10:  # Essentially zero range
        # Plot as a vertical line at the constant value
        constant_val = np.mean(clean_data)
        ax.axvline(constant_val, color=color, linestyle='-',
                   linewidth=2, alpha=alpha, label=label)
        # Add text annotation
        ylim = ax.get_ylim()
        ax.text(constant_val, ylim[1]*0.9, f'{constant_val:.3f}',
                rotation=90, verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.2))
        return

    # Try to plot histogram with decreasing bin counts if needed
    for n_bins in [30, 20, 15, 10]:
        try:
            ax.hist(clean_data, bins=n_bins, alpha=alpha,
                    label=label, density=True, color=color, edgecolor='none')
            return  # Success!
        except ValueError as e:
            if "Too many bins for data range" in str(e):
                continue  # Try fewer bins
            else:
                raise  # Re-raise unexpected errors

    # If we get here, even 2 bins failed - fall back to vertical line
    constant_val = np.mean(clean_data)
    ax.axvline(constant_val, color=color, linestyle='-',
               linewidth=2, alpha=alpha, label=label)
    ylim = ax.get_ylim()
    ax.text(constant_val, ylim[1]*0.9, f'{constant_val:.3f}',
            rotation=90, verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.2))

# Create PDF file in data directory
pdf_path = '../data/feature_distributions.pdf'
pdf = PdfPages(pdf_path)

# Get all feature names sorted by PSI (descending) - highest drift first
all_features = psi_df.sort_values('psi', ascending=False)['feature'].tolist()

# Configuration for plots per page
plots_per_page = 28  # 4 rows x 7 columns = 28 plots per page
n_features = len(all_features)
n_pages = (n_features + plots_per_page - 1) // plots_per_page  # Ceiling division

print(f"Plotting {n_features} features across {n_pages} pages ({plots_per_page} plots per page)")

# Create plots for each page
for page_num in range(n_pages):
    # Determine which features to plot on this page
    start_idx = page_num * plots_per_page
    end_idx = min(start_idx + plots_per_page, n_features)
    features_on_page = all_features[start_idx:end_idx]

    # Calculate grid dimensions
    n_cols = 7
    n_rows = (len(features_on_page) + n_cols - 1) // n_cols  # Ceiling division

    # Create subplot grid
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)

    # Flatten axes for easy iteration
    axes_flat = axes.flatten()

    # Plot each feature
    for idx, feature in enumerate(features_on_page):
        # Get data
        train_data = train_features_df[feature].dropna()
        test_data = test_features_df[feature].dropna()

        # Create histogram using safe function
        ax = axes_flat[idx]
        alpha = 0.7

        # Plot histograms using safe function
        safe_histogram(ax, train_data, 'Train', 'blue', alpha)
        safe_histogram(ax, test_data, 'Test', 'orange', alpha)

        # Add vertical lines for means (only if we have data)
        if len(train_data) > 0:
            ax.axvline(train_data.mean(), color='blue', linestyle='-', linewidth=1.5, alpha=0.8)
        if len(test_data) > 0:
            ax.axvline(test_data.mean(), color='orange', linestyle='-', linewidth=1.5, alpha=0.8)

        # Get PSI value for this feature
        psi_value = psi_df[psi_df["feature"]==feature]["psi"].values[0]

        # Formatting
        ax.set_title(f'{feature}\nPSI: {psi_value:.3f}', fontsize=9)
        ax.set_xlabel('Value', fontsize=8)
        ax.set_ylabel('Density', fontsize=8)
        ax.tick_params(axis='both', which='major', labelsize=7)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    # Hide unused subplots on this page
    for idx in range(len(features_on_page), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    # Add title to the figure
    fig.suptitle(f'Feature Distributions: Train vs Test (Page {page_num+1}/{n_pages})',
                 fontsize=14, y=0.98)

    # Adjust layout and save to PDF
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

# Close the PDF
pdf.close()

print(f"��✓ Saved feature distribution plots to: {os.path.abspath(pdf_path)}")
print(f"  Total pages: {n_pages}")
print(f"  Features per page: {plots_per_page}")
print(f"  Total features plotted: {n_features}")

# Also create a summary plot of PSI values and save it separately
print("\nCreating PSI summary plot...")
plt.figure(figsize=(14, 7))
psi_sorted = psi_df.sort_values('psi', ascending=False)
# Color code by PSI value
colors = []
for x in psi_sorted['psi']:
    if x >= 0.2:
        colors.append('red')
    elif x >= 0.1:
        colors.append('orange')
    else:
        colors.append('green')

bars = plt.bar(range(len(psi_sorted)), psi_sorted['psi'], color=colors, alpha=0.7, edgecolor='none')
plt.axhline(y=0.1, color='orange', linestyle='--', alpha=0.7, linewidth=1.5, label='Moderate change (PSI=0.1)')
plt.axhline(y=0.2, color='red', linestyle='--', alpha=0.7, linewidth=1.5, label='Significant change (PSI=0.2)')
plt.xlabel('Features (ranked by PSI - highest drift first)', fontsize=12)
plt.ylabel('Population Stability Index (PSI)', fontsize=12)
plt.title('Population Stability Index (PSI) for All Engineered Features\n'
          'Red: Significant drift (≥0.2) | Orange: Moderate drift (0.1-0.2) | Green: Minimal drift (<0.1)',
          fontsize=14, pad=20)
plt.legend(loc='upper right')
plt.xticks(rotation=90, fontsize=6)
plt.tight_layout()

# Save PSI summary plot
psi_plot_path = '../data/psi_summary_plot.png'
plt.savefig(psi_plot_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"��✓ Saved PSI summary plot to: {os.path.abspath(psi_plot_path)}")

# Print interpretation guide
print("\nPSI Interpretation Guide:")
print("- PSI < 0.1: No significant change")
print("- 0.1 ≤ PSI < 0.2: Moderate change")
print("- PSI ≥ 0.2: Significant change - indicates potential data drift")

# SHAP Feature Importance vs PSI Analysis

This section analyzes the relationship between feature importance (from SHAP values) and feature stability (PSI). Features with high SHAP importance and low PSI are ideal - they contribute significantly to model performance while being stable across datasets.

In [ ]:
# Load SHAP feature importance from the experiment
shap_importance_path = os.path.join(experiment_dir, 'features', 'shap_feature_importance.csv')
if os.path.exists(shap_importance_path):
    shap_importance_df = pd.read_csv(shap_importance_path)
    print(f"Loaded SHAP importance from: {shap_importance_path}")
    print(f"Shape: {shap_importance_df.shape}")
    print("Top 10 features by SHAP importance:")
    display(shap_importance_df.head(10))
else:
    print(f"SHAP importance file not found at: {shap_importance_path}")
    print("Please check the experiment directory path and ensure it contains:")
    print("  - features/shap_feature_importance.csv")
    # Create empty dataframe with expected columns to avoid errors downstream
    shap_importance_df = pd.DataFrame(columns=['feature', 'importance'])

In [ ]:
# Get PSI values from current analysis (psi_df should already exist from earlier in notebook)
# Ensure we have the PSI data
if 'psi_df' in locals():
    # Prepare PSI data for merging
    psi_for_merge = psi_df[['feature', 'psi']].copy()
    print(f"PSI data shape: {psi_for_merge.shape}")
else:
    print("PSI data not found. Please ensure PSI analysis has been run earlier in the notebook.")
    # Create empty dataframe to avoid errors downstream
    psi_for_merge = pd.DataFrame(columns=['feature', 'psi'])

# Merge SHAP importance with PSI values
if not shap_importance_df.empty and not psi_for_merge.empty:
    # Merge on feature name
    comparison_df = pd.merge(shap_importance_df, psi_for_merge, on='feature', how='inner')
    print(f"\nMerged dataset shape: {comparison_df.shape}")
    print(f"Features found in both datasets: {len(comparison_df)}")
    
    if len(comparison_df) > 0:
        print("\nTop 10 features by SHAP importance (with PSI):")
        display(comparison_df.sort_values('importance', ascending=False).head(10))
        
        print("\nTop 10 most stable features (lowest PSI):")
        display(comparison_df.sort_values('psi', ascending=True).head(10))
    else:
        print("No overlapping features found between SHAP importance and PSI data.")
else:
    print("Unable to create comparison due to missing data.")
    comparison_df = pd.DataFrame()

In [ ]:
print(comparison_df.to_string())

In [ ]:
# Create scatter plot: SHAP importance vs PSI
if not comparison_df.empty and len(comparison_df) > 0:
    plt.figure(figsize=(12, 8))
    
    # Create scatter plot
    scatter = plt.scatter(comparison_df['psi'], 
                         comparison_df['importance'],
                         alpha=0.7, 
                         s=100,
                         c=range(len(comparison_df)),  # Color by index for variety
                         cmap='viridis',
                         edgecolors='black',
                         linewidth=0.5)
    
    # Add color bar
    cbar = plt.colorbar(scatter)
    cbar.set_label('Feature Index', rotation=270, labelpad=20)
    
    # Add feature labels for points of interest (top 10 by importance and top 10 by stability)
    top_important = comparison_df.nlargest(10, 'importance')
    top_stable = comparison_df.nsmallest(10, 'psi')
    
    # Combine and remove duplicates
    labels_to_show = pd.concat([top_important, top_stable]).drop_duplicates()
    
    # Add labels for selected points
    for _, row in labels_to_show.iterrows():
        plt.annotate(row['feature'], 
                    (row['psi'], row['importance']),
                    xytext=(5, 5), 
                    textcoords='offset points',
                    fontsize=9,
                    alpha=0.8,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))
    
    # Add quadrant lines to help interpret the plot
    median_psi = comparison_df['psi'].median()
    median_importance = comparison_df['importance'].median()
    
    plt.axvline(x=median_psi, color='red', linestyle='--', alpha=0.7, label=f'Median PSI ({median_psi:.3f})')
    plt.axhline(y=median_importance, color='blue', linestyle='--', alpha=0.7, label=f'Median Importance ({median_importance:.3f})')
    
    # Add threshold lines for PSI interpretation
    plt.axvline(x=0.1, color='orange', linestyle=':', alpha=0.8, label='PSI=0.1 (moderate threshold)')
    plt.axvline(x=0.2, color='red', linestyle=':', alpha=0.8, label='PSI=0.2 (significant threshold)')
    
    # Set log scale for PSI (x-axis) to better visualize small PSI values
    plt.xscale('log')
    
    # Customize plot
    plt.xlabel('Population Stability Index (PSI) - Lower = More Stable (log scale)', fontsize=12)
    plt.ylabel('SHAP Feature Importance - Higher = More Important', fontsize=12)
    plt.title('SHAP Feature Importance vs Feature Stability (PSI)\n' +
              'Top-left: Important and stable (ideal)' +
              '\nTop-right: Important but unstable (may overfit)' +
              '\nBottom-left: Unimportant but stable (redundant?)' +
              '\nBottom-right: Unimportant and unstable (least valuable)', 
              fontsize=14, pad=20)
    
    plt.legend(loc='upper left', bbox_to_anchor=(0, 1))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print interpretation guide
    print("\n" + "="*60)
    print("INTERPRETATION GUIDE")
    print("="*60)
    print("Quadrant Analysis (based on medians):")
    print(f"  TOP-LEFT (High Importance, Low PSI):  Features important AND stable (IDEAL)")
    print(f"             Count: {len(comparison_df[(comparison_df['psi'] <= median_psi) & (comparison_df['importance'] > median_importance)])}")
    print(f"  TOP-RIGHT (High Importance, High PSI):  Features important but unstable")
    print(f"             Count: {len(comparison_df[(comparison_df['psi'] > median_psi) & (comparison_df['importance'] > median_importance)])}")
    print(f"  BOTTOM-LEFT (Low Importance, Low PSI):  Features unimportant but stable")
    print(f"             Count: {len(comparison_df[(comparison_df['psi'] <= median_psi) & (comparison_df['importance'] <= median_importance)])}")
    print(f"  BOTTOM-RIGHT (Low Importance, High PSI): Features unimportant and unstable")
    print(f"             Count: {len(comparison_df[(comparison_df['psi'] > median_psi) & (comparison_df['importance'] <= median_importance)])}")
    print("")
    print("PSI Thresholds:")
    print("  PSI < 0.1: Minimal drift (green)")
    print("  0.1 ≤ PSI < 0.2: Moderate drift (orange)")  
    print("  PSI ≥ 0.2: Significant drift (red)")
    print("="*60)
    
    # Save the plot
    plot_path = os.path.join(experiment_dir, 'shap_importance_vs_psi.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"\nPlot saved to: {plot_path}")
    
else:
    print("Cannot create scatter plot - no data available for comparison.")

## Summary: SHAP Feature Importance vs PSI Analysis

This analysis combines two critical perspectives for feature evaluation:

1. **SHAP Feature Importance**: Measures how much each feature contributes to model predictions (higher = more important for performance)
2. **PSI (Population Stability Index)**: Measures how much a feature's distribution shifts between train and test data (lower = more stable)

### Key Insights from the Analysis:

**Ideal Features (Top-Left Quadrant)**: 
- High SHAP importance + Low PSI (<0.1)
- These features are both important for model performance AND stable across datasets
- They generalize well and are reliable for production use

**Features Requiring Caution**:
- **Top-Right Quadrant**: High importance but high PSI (≥0.2)
  - May be overfitting to training data distributions
  - Performance may degrade significantly in production
- **Bottom-Left Quadrant**: Low importance but low PSI (<0.1)
  - Stable but contribute little to model performance
  - Could potentially be removed to simplify the model
- **Bottom-Right Quadrant**: Low importance and high PSI (≥0.2)
  - Neither important nor stable - prime candidates for removal

### Recommendations:

1. **Prioritize** features in the top-left quadrant for model development
2. **Investigate** features in the top-right quadrant - consider regularization or collecting more diverse training data
3. **Consider removing** features in the bottom-right quadrant (low value, high risk)
4. **Evaluate** features in the bottom-left quadrant - they may be safely removed to reduce model complexity

### Next Steps:
- Use these insights to inform feature selection strategies
- Consider building models with only the top-performing features from this analysis
- Monitor PSI values in production to detect when feature distributions begin to shift significantly

# Transfer Score Calculation

In [ ]:
# Merge all feature data
if not shap_importance_df.empty and not psi_df.empty and not results_df.empty:
    # Start with SHAP importance
    transfer_df = shap_importance_df.copy()
    
    # Merge with PSI data
    transfer_df = pd.merge(transfer_df, psi_df[['feature', 'psi']], on='feature', how='left')
    
    # Merge with statistical test results (KS and Cohen's d)
    transfer_df = pd.merge(transfer_df, results_df[['feature', 'ks_statistic', 'cohens_d']], on='feature', how='left')
    
    # Fill missing values with appropriate defaults
    transfer_df['psi'] = transfer_df['psi'].fillna(0)
    transfer_df['ks_statistic'] = transfer_df['ks_statistic'].fillna(0)
    transfer_df['cohens_d'] = transfer_df['cohens_d'].fillna(0)
    
    print(f"Merged dataset shape: {transfer_df.shape}")
    print(f"Features with complete data: {transfer_df.dropna().shape[0]}")
    
    # Calculate normalized values
    # Importance normalization (relative to maximum importance)
    max_importance = transfer_df['importance'].max()
    if max_importance > 0:
        transfer_df['importance_norm'] = transfer_df['importance'] / max_importance
    else:
        transfer_df['importance_norm'] = 0
    
    # PSI normalization: clip(psi / 2.0, 0, 1)
    # Dividing by 2.0 assumes PSI values typically won't exceed 2.0 for clipping to [0,1]
    transfer_df['psi_norm'] = np.clip(transfer_df['psi'] / 2.0, 0, 1)
    
    # KS normalization: KS statistic is already in [0,1] range
    transfer_df['ks_norm'] = transfer_df['ks_statistic']
    
    # Cohen's d normalization: clip(abs(cohen_d) / 2.0, 0, 1)
    # Dividing by 2.0 to map typical effect sizes to [0,1] range
    transfer_df['cohen_norm'] = np.clip(np.abs(transfer_df['cohens_d']) / 2.0, 0, 1)
    
    # Calculate shift index S with weights w1=0.4, w2=0.3, w3=0.3
    w1, w2, w3 = 0.6, 0.2, 0.2
    transfer_df['shift_index'] = (w1 * transfer_df['psi_norm'] + 
                                 w2 * transfer_df['ks_norm'] + 
                                 w3 * transfer_df['cohen_norm'])
    
    # Calculate transfer score: importance_norm / (1 + 2 * S)
    # Note: importance_norm**1 is just importance_norm
    transfer_df['transfer_score'] = transfer_df['importance_norm'] / (1 + 4 * transfer_df['shift_index'])
    #transfer_df['transfer_score'] = transfer_df['importance_norm'] * (1 - transfer_df['shift_index'])**2  # Emphasize stability by squaring (1 - S)
    
    # Sort by transfer score descending
    transfer_df_sorted = transfer_df.sort_values('transfer_score', ascending=False)
    
    # Display top 10 features by transfer score
    print("\nTop 20 Features by Transfer Score:")
    display_cols = ['feature', 'importance', 'importance_norm', 'psi', 'psi_norm', 
                   'ks_statistic', 'ks_norm', 'cohens_d', 'cohen_norm', 
                   'shift_index', 'transfer_score']
    display(transfer_df_sorted[display_cols].head(20))
    
    # Display bottom 10 features by transfer score
    print("\nBottom 10 Features by Transfer Score:")
    display(transfer_df_sorted[display_cols].tail(10))
    
    # Summary statistics
    print(f"\nTransfer Score Statistics:")
    print(f"  Mean: {transfer_df_sorted['transfer_score'].mean():.4f}")
    print(f"  Median: {transfer_df_sorted['transfer_score'].median():.4f}")
    print(f"  Std: {transfer_df_sorted['transfer_score'].std():.4f}")
    print(f"  Min: {transfer_df_sorted['transfer_score'].min():.4f}")
    print(f"  Max: {transfer_df_sorted['transfer_score'].max():.4f}")
    
    # Create visualization: Importance vs Transfer Score
    print("\nCreating Importance vs Transfer Score plot...")
    plt.figure(figsize=(10, 6))
    plt.scatter(transfer_df_sorted['importance'], transfer_df_sorted['transfer_score'], 
                alpha=0.7, s=50)
    plt.xlabel('SHAP Importance')
    plt.ylabel('Transfer Score')
    plt.title('Feature Importance vs Transfer Score\n(Higher transfer score = more important AND stable)')
    plt.grid(True, alpha=0.3)
    
    # Add feature labels for top 5 features
    top5 = transfer_df_sorted.head(5)
    for idx, row in top5.iterrows():
        plt.annotate(row['feature'], 
                    (row['importance'], row['transfer_score']),
                    xytext=(5, 5), 
                    textcoords='offset points',
                    fontsize=9,
                    alpha=0.8,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.3))
    
    plt.tight_layout()
    transfer_plot_path = os.path.join(experiment_dir, 'transfer_score_plot.png')
    plt.savefig(transfer_plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Transfer score plot saved to: {transfer_plot_path}")
    
    # Save transfer score results to CSV
    transfer_results_path = os.path.join(experiment_dir, 'transfer_score_results.csv')
    transfer_df_sorted.to_csv(transfer_results_path, index=False)
    print(f"Transfer score results saved to: {transfer_results_path}")
    
else:
    print("Unable to calculate transfer score - missing required data.")
    print(f"  SHAP data available: {not shap_importance_df.empty}")
    print(f"  PSI data available: {not psi_df.empty}")  
    print(f"  Statistical test data available: {not results_df.empty}")